# LangChain with OpenAI Tutorial: Prompt Engineering for Recipe Generation

This notebook provides a comprehensive guide to using LangChain with OpenAI's GPT models for prompt engineering, with a focus on culinary applications like recipe generation and meal planning.

## Table of Contents
1. [Setup and Installation](#setup)
2. [Basic LangChain and OpenAI Integration](#basic)
3. [Simple Recipe Generation](#simple)
4. [Advanced Prompt Engineering](#advanced)
5. [Creating a Weekly Meal Planner](#mealplan)
6. [Structured Output with LangChain](#structured)
7. [Exercises for Practice](#exercises)


<a id='setup'></a>
## 1. Setup and Installation

First, let's install the necessary libraries:

In [1]:
# Install required packages
!pip install langchain langchain_community openai python-dotenv

  Using cached tenacity-9.0.0-py3-none-any.whl.metadata (1.2 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached httpx_sse-0.4.0-py3-none-any.whl.metadata (9.0 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.3.2-py2.py3-none-any.whl.metadata (3.8 kB)
  Using cached marshmallow-3.26.1-py3-none-any.whl.metadata (7.3 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached mypy_extensions-1.0.0-py3-none-any.whl.metadata (1.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 9.3 MB/s eta 0:00:003 MB/s eta 0:00:01
  

Now, let's set up our environment and import the necessary libraries:

In [2]:
import os
from dotenv import load_dotenv
from langchain.llms import OpenAI
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts.chat import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from typing import List
from pydantic import BaseModel, Field

In [3]:
# Load environment variables from .env file
load_dotenv()

# Set your OpenAI API key
# Either store it in a .env file or set it directly here
#os.environ["OPENAI_API_KEY"] = ""

True

<a id='basic'></a>
## 2. Basic LangChain and OpenAI Integration

Let's start by initializing the ChatOpenAI model (GPT-3.5-Turbo) and creating a simple chain:

In [6]:
# Initialize the ChatOpenAI model
chat = ChatOpenAI(
    model="gpt-3.5-turbo",  # You can also use "gpt-4" if you have access
    temperature=0.7,  # Controls randomness: lower is more deterministic
    max_tokens=None,
    timeout=None,
    top_p=None,
    n=1,
    max_retries=2,
)

# Let's test our model
response = chat.predict("Briefly explain what prompt engineering is.")
print(response)

WARNING! top_p is not default parameter.
                    top_p was transferred to model_kwargs.
                    Please confirm that top_p is what you intended.


Prompt engineering is the process of designing and implementing prompts or cues within a system or environment to guide individuals towards desired behaviors or actions. This technique is commonly used in fields such as user experience design, product design, and behavior change interventions. By strategically placing prompts, designers can influence user decision-making and encourage specific behaviors.


<a id='simple'></a>
## 3. Simple Recipe Generation

Now, let's create our first prompt template for generating recipes:

In [7]:
# Create a template for recipe generation
recipe_template = PromptTemplate(
    input_variables=["ingredients", "cuisine", "dietary_restrictions"],
    template="""Create a detailed recipe using these ingredients: {ingredients}.
    The recipe should be {cuisine} style and consider these dietary restrictions: {dietary_restrictions}.
    Include cooking time, difficulty level, servings, ingredients with measurements, and step-by-step instructions."""
)

# Create a chain
recipe_chain = LLMChain(llm=chat, prompt=recipe_template)

# Generate a recipe
recipe = recipe_chain.run({
    "ingredients": "chicken, rice, bell peppers, onion",
    "cuisine": "Mediterranean",
    "dietary_restrictions": "low sodium"
})

print(recipe)

/var/folders/tl/zb5lzmw1379d6f96zdj440lm0000gn/T/ipykernel_31116/3075837724.py:10: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  recipe_chain = LLMChain(llm=chat, prompt=recipe_template)
/var/folders/tl/zb5lzmw1379d6f96zdj440lm0000gn/T/ipykernel_31116/3075837724.py:13: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  recipe = recipe_chain.run({


Recipe: Mediterranean Style Chicken and Rice with Bell Peppers and Onions

Cooking time: 45 minutes
Difficulty level: Medium
Servings: 4

Ingredients:
- 1 lb chicken breast, cut into bite-sized pieces
- 1 cup long-grain white rice
- 2 bell peppers (red, yellow, or orange), sliced
- 1 onion, sliced
- 2 cloves garlic, minced
- 1 tsp dried oregano
- 1 tsp dried basil
- 1/2 tsp paprika
- 1/4 tsp black pepper
- 2 cups low sodium chicken broth
- 2 tbsp olive oil
- Fresh parsley, chopped (for garnish)

Instructions:

1. In a large skillet, heat 1 tbsp olive oil over medium heat. Add the chicken pieces and cook until browned on all sides, about 5 minutes. Remove the chicken from the skillet and set aside.

2. In the same skillet, add the remaining 1 tbsp of olive oil. Add the sliced bell peppers, onion, and garlic. Saute for 5 minutes, until the vegetables are slightly softened.

3. Stir in the dried oregano, basil, paprika, and black pepper. Cook for another minute until the herbs are fragran

<a id='advanced'></a>
## 4. Advanced Prompt Engineering

Let's make our prompts more sophisticated using SystemMessagePromptTemplate and HumanMessagePromptTemplate:

In [9]:
# Create a system message that provides context and instruction
system_template = """You are a professional chef with expertise in {cuisine} cuisine.
Your task is to create delicious recipes that accommodate {dietary_restrictions} dietary needs.
Be creative but practical, using ingredients that are commonly available."""

system_message_prompt = SystemMessagePromptTemplate.from_template(system_template)

# Create a human message that provides the specific request
human_template = """Generate a detailed recipe using these ingredients: {ingredients}.
Please include:
1. Recipe title
2. Total preparation and cooking time
3. Difficulty level (Easy, Medium, Hard)
4. Number of servings
5. Ingredients with precise measurements
6. Step-by-step cooking instructions
7. Nutritional highlights
8. Serving suggestions"""

human_message_prompt = HumanMessagePromptTemplate.from_template(human_template)

# Combine the messages into a chat prompt template
chat_prompt = ChatPromptTemplate.from_messages([system_message_prompt, human_message_prompt])

# Create the chain
advanced_recipe_chain = LLMChain(llm=chat, prompt=chat_prompt)

# Generate a recipe
advanced_recipe = advanced_recipe_chain.run({
    "cuisine": "Italian",
    "dietary_restrictions": "vegetarian",
    "ingredients": "pasta, tomatoes, basil, garlic, olive oil, mozzarella"
})

print(advanced_recipe)

Recipe: Caprese Pasta Salad

Total preparation and cooking time: 30 minutes
Difficulty level: Easy
Number of servings: 4

Ingredients:
- 8 oz pasta (such as penne or fusilli)
- 2 cups cherry tomatoes, halved
- 1/2 cup fresh basil leaves, chopped
- 3 cloves garlic, minced
- 1/4 cup olive oil
- 8 oz fresh mozzarella, diced
- Salt and pepper to taste

Step-by-step cooking instructions:
1. Cook the pasta according to package instructions until al dente. Drain and rinse under cold water to stop the cooking process. Set aside.
2. In a large mixing bowl, combine the cherry tomatoes, basil, garlic, olive oil, and mozzarella. Season with salt and pepper to taste.
3. Add the cooked pasta to the bowl and gently toss everything together until well combined.
4. Taste and adjust seasoning if needed.
5. Serve the Caprese Pasta Salad chilled or at room temperature.

Nutritional highlights:
- This Caprese Pasta Salad is a light and refreshing dish that is rich in antioxidants from the tomatoes and basi

<a id='mealplan'></a>
## 5. Creating a Weekly Meal Planner

Now, let's build a more complex application: a weekly meal planner!

In [11]:
# Create a system message for the meal planner
meal_planner_system = """You are a professional nutritionist and meal planning expert.
Your task is to create a balanced, healthy weekly meal plan that fits the user's dietary preferences,
restrictions, and calorie goals. Each day should include breakfast, lunch, dinner, and a snack."""

meal_planner_system_prompt = SystemMessagePromptTemplate.from_template(meal_planner_system)

# Create a human message for the specific meal plan request
meal_planner_human = """Create a 7-day meal plan with the following criteria:
- Dietary preference: {diet_type}
- Allergies/restrictions: {restrictions}
- Daily calorie target: {calorie_target}
- Cooking complexity preference: {cooking_complexity}
- Number of people: {people_count}

For each day, provide:
1. Day of the week
2. Breakfast (with brief recipe)
3. Lunch (with brief recipe)
4. Dinner (with brief recipe)
5. Snack option
6. Estimated calorie count for each meal
7. A grocery list for the entire week at the end"""

meal_planner_human_prompt = HumanMessagePromptTemplate.from_template(meal_planner_human)

# Combine into a chat prompt template
meal_planner_prompt = ChatPromptTemplate.from_messages([meal_planner_system_prompt, meal_planner_human_prompt])

# Create the chain
meal_planner_chain = LLMChain(llm=chat, prompt=meal_planner_prompt)

# Generate a meal plan
meal_plan = meal_planner_chain.run({
    "diet_type": "Mediterranean",
    "restrictions": "no shellfish, low sodium",
    "calorie_target": "2000",
    "cooking_complexity": "moderate, with some quick options for busy days",
    "people_count": "2"
})

print(meal_plan)

**7-Day Mediterranean Meal Plan:**

**Day 1:**
1. **Breakfast:** Greek Yogurt Parfait with honey, walnuts, and berries.
2. **Lunch:** Mediterranean Quinoa Salad with cucumbers, cherry tomatoes, feta cheese, and a lemon vinaigrette.
3. **Dinner:** Baked Lemon Herb Chicken with roasted vegetables (bell peppers, zucchini, and onions).
4. **Snack:** Hummus with carrot sticks.
5. **Calories:** Breakfast - 300, Lunch - 400, Dinner - 500, Snack - 200

**Day 2:**
1. **Breakfast:** Whole grain toast topped with avocado, cherry tomatoes, and feta cheese.
2. **Lunch:** Greek Chickpea Salad with cucumbers, red onions, olives, and a Greek dressing.
3. **Dinner:** Grilled Salmon with a Mediterranean salsa (tomatoes, olives, capers) and quinoa.
4. **Snack:** Mixed nuts.
5. **Calories:** Breakfast - 350, Lunch - 450, Dinner - 550, Snack - 250

**Day 3:**
1. **Breakfast:** Shakshuka (poached eggs in a tomato and pepper sauce).
2. **Lunch:** Mediterranean Stuffed Bell Peppers with ground turkey, quinoa,

<a id='structured'></a>
## 6. Structured Output with LangChain

Now, let's make our output more structured and usable in applications by using Pydantic models and output parsers:

In [10]:
# Define a Pydantic model for a recipe
class Ingredient(BaseModel):
    name: str = Field(description="The name of the ingredient")
    quantity: str = Field(description="The quantity of the ingredient needed")
    unit: str = Field(description="The unit of measurement for the ingredient")

class RecipeStep(BaseModel):
    step_number: int = Field(description="The step number in the cooking process")
    instruction: str = Field(description="The detailed cooking instruction for this step")

class Recipe(BaseModel):
    title: str = Field(description="The title of the recipe")
    cooking_time: str = Field(description="Total time required to prepare and cook")
    difficulty: str = Field(description="Difficulty level (Easy, Medium, Hard)")
    servings: int = Field(description="Number of servings this recipe makes")
    ingredients: List[Ingredient] = Field(description="List of ingredients with quantities")
    instructions: List[RecipeStep] = Field(description="Step-by-step cooking instructions")
    nutritional_info: str = Field(description="Brief nutritional highlights")
    serving_suggestion: str = Field(description="Suggestion on how to serve the dish")

# Create a parser for the recipe
recipe_parser = PydanticOutputParser(pydantic_object=Recipe)

# Create a format instruction
format_instructions = recipe_parser.get_format_instructions()

# Create a structured recipe prompt
structured_recipe_template = """You are a professional chef who specializes in creating delicious recipes.
Create a {cuisine} recipe using these ingredients: {ingredients}.
Consider these dietary restrictions: {dietary_restrictions}.

{format_instructions}
"""

structured_prompt = PromptTemplate(
    template=structured_recipe_template,
    input_variables=["cuisine", "ingredients", "dietary_restrictions"],
    partial_variables={"format_instructions": format_instructions}
)

# Create a chain for structured output
structured_recipe_chain = LLMChain(llm=chat, prompt=structured_prompt)

# Generate a structured recipe
structured_output = structured_recipe_chain.run({
    "cuisine": "Japanese",
    "ingredients": "salmon, rice, nori seaweed, avocado, cucumber, soy sauce",
    "dietary_restrictions": "gluten-free"
})

print(structured_output)

# Parse the structured output into a Recipe object
try:
    recipe_object = recipe_parser.parse(structured_output)
    print("\nParsed Recipe Object:")
    print(f"Title: {recipe_object.title}")
    print(f"Cooking Time: {recipe_object.cooking_time}")
    print(f"Difficulty: {recipe_object.difficulty}")
    print(f"Servings: {recipe_object.servings}")

    print("\nIngredients:")
    for ingredient in recipe_object.ingredients:
        print(f"- {ingredient.quantity} {ingredient.unit} {ingredient.name}")

    print("\nInstructions:")
    for step in recipe_object.instructions:
        print(f"{step.step_number}. {step.instruction}")

    print(f"\nNutritional Info: {recipe_object.nutritional_info}")
    print(f"Serving Suggestion: {recipe_object.serving_suggestion}")

except Exception as e:
    print(f"Error parsing recipe: {e}")

{
  "title": "Salmon Avocado Sushi Bowl",
  "cooking_time": "30 minutes",
  "difficulty": "Easy",
  "servings": 2,
  "ingredients": [
    {
      "name": "Salmon",
      "quantity": "200g",
      "unit": "grams"
    },
    {
      "name": "Rice",
      "quantity": "1 cup",
      "unit": "cup"
    },
    {
      "name": "Nori seaweed",
      "quantity": "2 sheets",
      "unit": "sheets"
    },
    {
      "name": "Avocado",
      "quantity": "1",
      "unit": "piece"
    },
    {
      "name": "Cucumber",
      "quantity": "1/2",
      "unit": "piece"
    },
    {
      "name": "Soy sauce",
      "quantity": "2 tbsp",
      "unit": "tablespoon"
    }
  ],
  "instructions": [
    {
      "step_number": 1,
      "instruction": "Cook the rice according to package instructions and let it cool."
    },
    {
      "step_number": 2,
      "instruction": "Cut the salmon, avocado, and cucumber into bite-sized pieces."
    },
    {
      "step_number": 3,
      "instruction": "Lay out a sheet 

<a id='exercises'></a>
## 7. Exercises for Practice

Now it's your turn! Complete the following exercises to practice your LangChain and prompt engineering skills.

### Exercise 1: Recipe Generator with Specific Constraints

Create a prompt that generates recipes with the following constraints:
- Must use exactly 5 ingredients (no more, no less)
- Must be cooked in 30 minutes or less
- Must be suitable for beginners
- Should include a creative title

Test your prompt with at least two different cuisine types.

In [ ]:
# Your code here
# Create a prompt template for generating quick 5-ingredient recipes



## Conclusion

In this notebook, we've explored how to use LangChain with OpenAI's GPT models to create powerful prompt engineering solutions for recipe generation and meal planning. We've covered:

- Basic integration of LangChain with OpenAI
- Simple and advanced prompt templates
- Structured output using Pydantic models
- Complex applications like meal planning

Prompt engineering is both an art and a science. The key is to experiment with different approaches, be specific in your instructions, and provide clear context to the model. As you complete the exercises, pay attention to how small changes in your prompts can lead to significant differences in the output.

Happy prompting!